# Segmentation fine-tuning on Kvasir-SEG

SegFormer all-MLP decoder on a ViT simple feature pyramid, 352px, 100 epochs.
Five encoders x five seeds = 25 runs, ~4 GPU-hours total.

Every arm uses the **identical** decoder, recipe, splits and seeds — only the encoder weights differ. Test is scored exactly once, on the best-val checkpoint.

In [ ]:
REPO_URL = "https://github.com/morsalin101/jepa-thesis.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/jepa-thesis"


In [ ]:
import os, subprocess, sys

def sh(cmd, check=True):
    """Run a shell command, streaming its output live.

    Streaming rather than capture_output matters because these jobs run for
    tens of minutes and report progress as they go. Buffering that until the
    process exits makes a long job indistinguishable from a hung one.
    """
    print('$', cmd, flush=True)
    # PYTHONUNBUFFERED: a child process writing to a pipe switches from line
    # buffering to 4 KB block buffering, so progress lines would still arrive
    # in bursts (or not at all until exit) even though we stream them here.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code_ = p.wait()
    if check and code_ != 0:
        # Include the tail of the output in the exception. Otherwise the
        # traceback shows only this wrapper and the real error is buried
        # further up the cell, which is easy to miss and impossible to
        # copy/paste usefully.
        tail = ''.join(lines[-25:]).rstrip()
        raise RuntimeError(
            f'command failed (exit {code_}): {cmd}\n\n--- last output ---\n{tail}')
    return subprocess.CompletedProcess(cmd, code_, ''.join(lines), '')

if subprocess.run(f'git ls-remote {REPO_URL}', shell=True,
                  capture_output=True).returncode != 0:
    raise RuntimeError('Cannot reach GitHub — turn Internet ON in the session options.')

if os.path.exists(WORKDIR):
    sh(f'cd {WORKDIR} && git fetch origin && git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {REPO_URL} {WORKDIR}')
sh(f'cd {WORKDIR} && git log -1 --oneline')
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)


In [ ]:
sh('pip install -q -r requirements.txt')


In [ ]:
# Report the accelerator and the precision that follows from it.
# T4 (sm_75) has fp16 tensor cores but NO bf16 hardware; P100 (sm_60) has
# neither and runs ~2x slower. The code adapts either way — this cell is
# here so you know what you were given before spending 8 hours on it.
import torch
from src.config import amp_config
print('CUDA devices:', torch.cuda.device_count())
amp = amp_config()
print(amp)
if torch.cuda.device_count() < 2:
    print('\n*** Only one GPU. I-JEPA and MAE will still run correctly, but\n'
          '    SimCLR/MoCo v3 need 2 GPUs to preserve global_batch=512.\n'
          '    Set Session options -> Accelerator -> GPU T4 x2 and re-run. ***')


In [ ]:
# Pull the exported encoders published by the pretraining notebooks.
# Recursive: Kaggle nests mounts as /kaggle/input/datasets/<owner>/<slug>/,
# so a fixed path finds nothing and segmentation then reports the confusing
# 'run pretraining first' for a run that already finished.
import shutil, glob, pathlib
dst = pathlib.Path('/kaggle/working/weights')
dst.mkdir(parents=True, exist_ok=True)
found = [p for p in glob.glob('/kaggle/input/**/*.pt', recursive=True)
         if 'jepa-thesis-weights' in p]
for p in found:
    shutil.copy(p, dst)
print('encoders available:', sorted(f.name for f in dst.glob('*.pt')) or 'NONE')
if not found:
    print('Attach the jepa-thesis-weights dataset via + Add Input. '
          '(Only the `random` arm can run without it.)')


In [ ]:
# ---- run configuration -------------------------------------------------
# Only list encoders you have actually pretrained — check the cell above.
# 'random' needs no weights and is the control arm: it shows what the decoder
# alone achieves, so any pretraining gain is measured against it.
ENCODERS = ['ijepa', 'random']
SEEDS    = [0]        # [0,1,2,3,4] for the final results table
EPOCHS   = 30         # 100 for the final table
RUN_ABLATIONS = False # low-label / decoder-swap / 880-120 (adds many runs)
# ------------------------------------------------------------------------

n = len(ENCODERS) * len(SEEDS)
print(f'{n} run(s) x ~{EPOCHS*0.25:.0f} min = ~{n*EPOCHS*0.25:.0f} min total')
if n * EPOCHS * 0.25 > 420:
    print('*** This exceeds a 7.5h session. Cut SEEDS or EPOCHS. ***')


In [ ]:
for enc in ENCODERS:
    for seed in SEEDS:
        sh(f'python -m src.engine.segment --encoder {enc} --seed {seed} --epochs {EPOCHS}', check=False)


In [ ]:
# Test-set results for every completed run. Reads summary.json, so it works
# even if some arms failed or the session was cut short.
import json, glob

rows = []
for p in sorted(glob.glob('/kaggle/working/seg/*/summary.json')):
    rows.append(json.load(open(p)))

if not rows:
    print('no completed runs yet')
else:
    hdr = f"{'encoder':<10} {'seed':>4} {'Dice':>7} {'IoU':>7} {'HD95':>7} "\
          f"{'fail%':>6} {'val':>7} {'ep':>4} {'min':>6}"
    print(hdr); print('-' * len(hdr))
    for r in sorted(rows, key=lambda r: (-r['dice'])):
        print(f"{r['encoder']:<10} {r['seed']:>4} {r['dice']:>7.4f} "
              f"{r['iou']:>7.4f} {r['hd95']:>7.1f} {r['failure_rate']*100:>5.0f}% "
              f"{r['best_val_dice']:>7.4f} {r['best_epoch']:>4} {r['train_minutes']:>6.1f}")

    if len(rows) > 1:
        best = max(rows, key=lambda r: r['dice'])
        ctrl = [r for r in rows if r['encoder'] == 'random']
        if ctrl:
            d = best['dice'] - ctrl[0]['dice']
            print(f"\nbest ({best['encoder']}) vs random init: {d:+.4f} Dice")
            print('A small gap is expected when the encoder was pretrained on a '
                  'subset — SSL needs corpus scale.')


In [ ]:
# Training curves for each arm.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
any_data = False
for p in sorted(glob.glob('/kaggle/working/seg/*/metrics.jsonl')):
    recs = [json.loads(l) for l in open(p) if l.strip()]
    if not recs:
        continue
    any_data = True
    lab = f"{recs[0]['encoder']} s{recs[0]['seed']}"
    ax[0].plot([r['epoch'] for r in recs], [r['train_loss'] for r in recs], label=lab)
    ax[1].plot([r['epoch'] for r in recs], [r['val_dice'] for r in recs], label=lab)
if any_data:
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('train loss'); ax[0].grid(alpha=.3)
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('val Dice'); ax[1].grid(alpha=.3)
    ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('no metrics yet')


In [ ]:
# Qualitative grid + all other figures, written to /kaggle/working/figures.
sh('python -m src.viz.make_all --out /kaggle/working/figures', check=False)

from IPython.display import Image as IPyImage, display
for f in sorted(glob.glob('/kaggle/working/figures/*.png')):
    print(os.path.basename(f)); display(IPyImage(filename=f))


## Ablations

Only runs when `RUN_ABLATIONS = True`. Each block multiplies the run count, so enable them once the main comparison is settled — not before.

In [ ]:
if RUN_ABLATIONS:
    # Low-label regime — usually the strongest result, since SSL differences
    # are largest where labels are scarce.
    for enc in ENCODERS:
        for lf in [0.1, 0.25, 0.5]:
            for seed in SEEDS:
                sh(f'python -m src.engine.segment --encoder {enc} --label-fraction {lf} --seed {seed} --epochs {EPOCHS}', check=False)

    # Does the encoder ranking survive a decoder swap?
    for enc in ENCODERS:
        sh(f'python -m src.engine.segment --encoder {enc} --decoder unet --seed 0 --epochs {EPOCHS}', check=False)

    # 880/120 split, for comparability with published Kvasir-SEG numbers.
    for enc in ENCODERS:
        sh(f'python -m src.engine.segment --encoder {enc} --split 880_120 --seed 0 --epochs {EPOCHS}', check=False)
else:
    print('ablations disabled — set RUN_ABLATIONS = True to enable')
